# Exp47 — CRAFT matched retrieval-rule comparison on the frozen 24-condition benchmark

**Kernel:** `Python 3.12 (Stock Regime)`

This notebook tests whether the main PRR result depends on using Pearson as the conventional retrieval baseline by adding a recent retrieval rule from:

> Kang et al., **Channel-wise Retrieval for Multivariate Time Series Forecasting (CRAFT)**, ICASSP 2026 / arXiv:2604.05543v2.

CRAFT uses two retrieval ingredients:

1. a **train-only sparse channel relation graph** based on cosine similarity between concatenated historical channel trajectories;
2. **low-frequency complex FFT similarity** for ranking candidate historical keys.

The paper uses \(L=720\), \(M=3\) graph neighbors, \(F=36\) low-frequency FFT coefficients (5% of \(L\)), and one retrieved reference.  
Our frozen PRR benchmark uses \(L=96\), memory stride 8, and evaluates the final **Top-10** by AnalogFutureMSE. To keep the retrieval metric identical across methods, the primary comparison therefore uses \(F=\mathrm{round}(0.05L)=5\) and Top-10.

We report two carefully labeled diagnostics:

- **CRAFT-SC@10 (matched):** CRAFT Eq. (2) spectral retrieval restricted to the **same-channel memory**, exactly matching PRR's candidate-support protocol.
- **CRAFT-Graph@10:** CRAFT Eq. (1)+(2), using a train-only top-\(M=3\) channel graph and searching historical keys from those graph channels. This is closer to CRAFT's channel-wise retrieval mechanism, but its candidate support is cross-channel and therefore is reported separately.

This is a **matched retrieval-rule diagnostic**, not a reproduction of CRAFT's end-to-end forecasting benchmark. No CRAFT direct forecaster, additive \(\alpha\), or forecasting head is trained here.

The notebook additionally reports the 16 conditions on datasets overlapping the CRAFT paper
(ETTh1, Electricity, Traffic, Weather) separately from the full six-dataset / 24-condition benchmark.


In [1]:
# Cell 1 — Stock Regime kernel guard and environment
import sys, os, json, time, gc, math
from pathlib import Path

import numpy as np
import pandas as pd

print("Python executable:", sys.executable)
print("Python version   :", sys.version.split()[0])

if sys.version_info < (3, 12):
    raise RuntimeError(
        "Wrong Jupyter kernel. Select 'Python 3.12 (Stock Regime)' / 'Stock Regime' "
        "before running this notebook."
    )

try:
    import torch
    print("PyTorch          :", torch.__version__)
    print("CUDA available   :", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU              :", torch.cuda.get_device_name(0))
except Exception as e:
    torch = None
    print("PyTorch unavailable:", repr(e))

print("NumPy            :", np.__version__)
print("[PASS] Stock Regime-compatible kernel detected")


Python executable: /data/envs/stock_regime/bin/python
Python version   : 3.12.13
PyTorch          : 2.12.1+cu126
CUDA available   : True
GPU              : NVIDIA A100-SXM4-80GB
NumPy            : 2.4.6
[PASS] Stock Regime-compatible kernel detected


In [2]:
# Cell 2 — experiment configuration
DATASETS = ("ETTh1", "Weather", "Electricity", "Traffic", "Exchange", "Solar")
HORIZONS = (96, 192, 336, 720)
CRAFT_PAPER_OVERLAP = ("ETTh1", "Weather", "Electricity", "Traffic")

L = 96
MEMORY_STRIDE = 8
TOP_K = 10

# CRAFT paper: M=3 related channels; F=36 when L=720 (=5% of L).
CRAFT_GRAPH_M = 3
CRAFT_FREQ_RATIO = 0.05
CRAFT_F = max(1, int(round(CRAFT_FREQ_RATIO * L)))
CRAFT_EPS = 1e-8

# The public paper text does not explicitly state whether the channel itself is removed
# before top-M graph selection. We keep self eligible by default (literal top-M cosine).
# Set False for an optional sensitivity analysis, but do not mix the two in the main table.
GRAPH_INCLUDE_SELF = True

QUERY_CHUNK = 128
RUN_GRAPH_VARIANT = True
RUN_SMOKE_TEST = True
RECOMPUTE = False

print("L / stride / TopK :", L, MEMORY_STRIDE, TOP_K)
print("CRAFT M / F       :", CRAFT_GRAPH_M, CRAFT_F)
print("Graph self policy :", "include self" if GRAPH_INCLUDE_SELF else "exclude self")


L / stride / TopK : 96 8 10
CRAFT M / F       : 3 5
Graph self policy : include self


In [3]:
# Cell 3 — locate the exact Exp33 frozen manifests and output folder
manifest_candidates = [
    Path("/data/code/which_histories_matter_ext/results/exp33/final_eval_manifest"),
    Path("/data/code/stock_regime_retrieval/strong_forecaster/results/exp33/final_eval_manifest"),
]

for base in [Path("/data/code"), Path("/data")]:
    if base.exists():
        manifest_candidates += list(base.glob("*/results/exp33/final_eval_manifest"))
        manifest_candidates += list(base.glob("*/*/results/exp33/final_eval_manifest"))

MANIFEST_ROOT = next((p for p in manifest_candidates if p.exists()), None)
assert MANIFEST_ROOT is not None, "Exp33 frozen manifests not found."

PROJECT_ROOT = MANIFEST_ROOT.parents[2]
OUT_ROOT = PROJECT_ROOT / "results" / "exp47_craft_matched_retrieval"
CACHE_ROOT = OUT_ROOT / "cache"
GRAPH_ROOT = OUT_ROOT / "graphs"

for p in (OUT_ROOT, CACHE_ROOT, GRAPH_ROOT):
    p.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "datasets": list(DATASETS),
    "horizons": list(HORIZONS),
    "lookback": L,
    "memory_stride": MEMORY_STRIDE,
    "top_k": TOP_K,
    "craft_graph_m": CRAFT_GRAPH_M,
    "craft_f": CRAFT_F,
    "craft_freq_ratio": CRAFT_FREQ_RATIO,
    "graph_include_self": GRAPH_INCLUDE_SELF,
    "note": "CRAFT matched retrieval-rule diagnostic; not end-to-end CRAFT forecasting reproduction.",
}
(OUT_ROOT / "config.json").write_text(json.dumps(CONFIG, indent=2), encoding="utf-8")

print("MANIFEST_ROOT:", MANIFEST_ROOT)
print("PROJECT_ROOT :", PROJECT_ROOT)
print("OUT_ROOT     :", OUT_ROOT)


MANIFEST_ROOT: /data/code/which_histories_matter_ext/results/exp33/final_eval_manifest
PROJECT_ROOT : /data/code/which_histories_matter_ext
OUT_ROOT     : /data/code/which_histories_matter_ext/results/exp47_craft_matched_retrieval


In [4]:
# Cell 4 — dataset loader; identical normalization/splits to the frozen PRR retrieval benchmark
EXPECTED = {
    "ETTh1": 7,
    "Weather": 21,
    "Electricity": 321,
    "Traffic": 862,
    "Exchange": 8,
    "Solar": 137,
}

CANDIDATES = {
    "ETTh1": [
        "/data/Time-Series-Library/dataset/ETT-small/ETTh1.csv",
        "/data/Time-Series-Library_v2/dataset/ETT-small/ETTh1.csv",
        "/data/dataset/ETT-small/ETTh1.csv",
        "/data/dataset/ETTh1.csv",
    ],
    "Weather": [
        "/data/Time-Series-Library/dataset/weather/weather.csv",
        "/data/Time-Series-Library_v2/dataset/weather/weather.csv",
        "/data/dataset/weather/weather.csv",
        "/data/dataset/weather.csv",
    ],
    "Electricity": [
        "/data/Time-Series-Library/dataset/electricity/electricity.csv",
        "/data/Time-Series-Library_v2/dataset/electricity/electricity.csv",
        "/data/dataset/electricity/electricity.csv",
        "/data/dataset/electricity.csv",
    ],
    "Traffic": [
        "/data/Time-Series-Library/dataset/traffic/traffic.csv",
        "/data/Time-Series-Library_v2/dataset/traffic/traffic.csv",
        "/data/dataset/traffic/traffic.csv",
        "/data/dataset/traffic.csv",
    ],
    "Exchange": [
        "/data/Time-Series-Library/dataset/exchange_rate/exchange_rate.csv",
        "/data/Time-Series-Library_v2/dataset/exchange_rate/exchange_rate.csv",
        "/data/dataset/exchange_rate/exchange_rate.csv",
        "/data/dataset/exchange_rate.csv",
    ],
    "Solar": [
        "/data/Time-Series-Library/dataset/Solar/solar_AL.txt",
        "/data/Time-Series-Library_v2/dataset/Solar/solar_AL.txt",
        "/data/dataset/Solar/solar_AL.txt",
        "/data/dataset/solar_AL.txt",
    ],
}

def resolve_dataset(name):
    for s in CANDIDATES[name]:
        p = Path(s)
        if p.is_file():
            return p
    raise FileNotFoundError(f"Dataset not found: {name}")

def clean_numeric(df):
    return (
        df.apply(pd.to_numeric, errors="coerce")
          .dropna(axis=1, how="all")
          .replace([np.inf, -np.inf], np.nan)
          .interpolate(axis=0, limit_direction="both")
          .ffill()
          .bfill()
    )

def load_data(name):
    p = resolve_dataset(name)

    if name == "Solar":
        df = pd.read_csv(p, header=None)
        if df.shape[1] == 1:
            df = pd.read_csv(p, header=None, sep=r"\s+")
        num = clean_numeric(df)
    else:
        df = pd.read_csv(p)
        date_col = next(
            (c for c in df.columns if str(c).lower() in {"date", "datetime", "timestamp", "time"}),
            None,
        )
        val = df.drop(columns=[date_col]) if date_col is not None else df.copy()

        # Preserve the ordering used by the earlier frozen benchmark loader.
        if name in {"Weather", "Electricity"} and "OT" in val.columns:
            val = val[[c for c in val.columns if c != "OT"] + ["OT"]]

        num = clean_numeric(val)

    raw = num.to_numpy(np.float32)
    assert raw.shape[1] == EXPECTED[name], (name, raw.shape)

    n = len(raw)
    if name == "ETTh1":
        assert raw.shape == (17420, 7)
        train_end, val_end = 8640, 11520
    else:
        train_end = int(0.70 * n)
        val_end = n - int(0.20 * n)

    mu = raw[:train_end].mean(0).astype(np.float32)
    sd = raw[:train_end].std(0, ddof=0).astype(np.float32)
    assert np.all(sd > 1e-6), (name, float(sd.min()))

    z = ((raw - mu) / sd).astype(np.float32)

    return {
        "path": p,
        "raw": raw,
        "z": z,
        "n": n,
        "channels": raw.shape[1],
        "train_end": train_end,
        "val_end": val_end,
    }

DATA = {d: load_data(d) for d in DATASETS}

display(pd.DataFrame([
    {
        "dataset": d,
        "rows": DATA[d]["n"],
        "channels": DATA[d]["channels"],
        "train_end": DATA[d]["train_end"],
        "val_end": DATA[d]["val_end"],
        "path": str(DATA[d]["path"]),
    }
    for d in DATASETS
]))


,dataset,rows,channels,train_end,val_end,path
0,ETTh1,17420,7,8640,11520,/data/Time-Series-Library/dataset/ETT-small/ET...
1,Weather,52696,21,36887,42157,/data/Time-Series-Library/dataset/weather/weat...
2,Electricity,26304,321,18412,21044,/data/Time-Series-Library/dataset/electricity/...
3,Traffic,17544,862,12280,14036,/data/Time-Series-Library/dataset/traffic/traf...
4,Exchange,7588,8,5311,6071,/data/Time-Series-Library/dataset/exchange_rat...
5,Solar,52560,137,36792,42048,/data/Time-Series-Library/dataset/Solar/solar_...


In [5]:
# Cell 5 — load and audit the frozen query/channel manifests
def load_manifest(dataset, horizon, split):
    p = MANIFEST_ROOT / f"{dataset}_H{horizon}_{split}.npz"
    assert p.exists(), p
    with np.load(p, allow_pickle=False) as z:
        anchor = np.asarray(z["anchor"], np.int64)
        channel = np.asarray(z["channel"], np.int64)

    assert anchor.shape == channel.shape
    assert len(anchor) > 0
    assert channel.min() >= 0 and channel.max() < DATA[dataset]["channels"]
    return anchor, channel

audit = []
for d in DATASETS:
    for h in HORIZONS:
        for split in ("val", "test"):
            a, c = load_manifest(d, h, split)
            audit.append({
                "dataset": d,
                "horizon": h,
                "split": split,
                "pairs": len(a),
                "unique_channels": len(np.unique(c)),
            })

audit_df = pd.DataFrame(audit)
display(audit_df)
audit_df.to_csv(OUT_ROOT / "manifest_audit.csv", index=False)


,dataset,horizon,split,pairs,unique_channels
0,ETTh1,96,val,4096,7
1,ETTh1,96,test,8192,7
2,ETTh1,192,val,4096,7
3,ETTh1,192,test,8192,7
4,ETTh1,336,val,4096,7
5,ETTh1,336,test,8192,7
6,ETTh1,720,val,4096,7
7,ETTh1,720,test,8192,7
8,Weather,96,val,4096,21
9,Weather,96,test,8192,21


## CRAFT retrieval implementation used here

For a query channel \(c\), CRAFT first constructs a train-only channel graph using

\[
\operatorname{sim}_{\rm graph}(i,j)
=
\frac{z_i^\top z_j}{\|z_i\|\|z_j\|},
\]

where each \(z_i\) is the concatenation of the channel's historical key windows.

Rather than explicitly materializing this very long concatenated vector, the code below computes the **exact same dot products** with time-point multiplicity weights: each training time point is weighted by the number of stride-8 key windows containing it.

For candidate ranking, the first \(F=5\) complex FFT coefficients are retained and scored by the normalized real complex inner product,

\[
\operatorname{sim}_{\rm FFT}(q,k)
=
\frac{\Re(q k^\ast)}
{\|q\|\,\|k\|+\epsilon}.
\]

No future values enter either graph construction or retrieval scoring.


In [6]:
# Cell 6 — shared window/future utilities and exact CRAFT graph construction
def past_windows(z, ch, anchors):
    anchors = np.asarray(anchors, np.int64)
    idx = anchors[:, None] - L + np.arange(L)[None, :]
    return z[idx, int(ch)].astype(np.float32)

def future_residuals(z, ch, anchors, horizon):
    anchors = np.asarray(anchors, np.int64)
    idx = anchors[:, None] + np.arange(horizon)[None, :]
    fut = z[idx, int(ch)].astype(np.float32)
    cur = z[anchors - 1, int(ch)].astype(np.float32)
    return (fut - cur[:, None]).astype(np.float32)

def future_residuals_variable_channel(z, channels, anchors, horizon):
    channels = np.asarray(channels, np.int64)
    anchors = np.asarray(anchors, np.int64)
    assert channels.shape == anchors.shape
    offs = np.arange(horizon, dtype=np.int64)
    idx_t = anchors[:, None] + offs[None, :]
    idx_c = channels[:, None]
    fut = z[idx_t, idx_c].astype(np.float32)
    cur = z[anchors - 1, channels].astype(np.float32)
    return (fut - cur[:, None]).astype(np.float32)

def craft_fft_features(x):
    """
    CRAFT Eq.(2) feature: first F coefficients of the complex FFT,
    normalized to unit complex L2 norm.
    """
    x = np.asarray(x, np.float32)
    spec = np.fft.fft(x, axis=-1)[..., :CRAFT_F].astype(np.complex64)
    norm = np.sqrt(np.sum(np.abs(spec) ** 2, axis=-1, keepdims=True)).astype(np.float32)
    return spec / np.maximum(norm, CRAFT_EPS)

def craft_similarity(qspec, kspec):
    # qspec: [B,F], kspec: [N,F], both unit normalized
    return np.real(qspec @ np.conjugate(kspec).T).astype(np.float32)

def valid_memory_anchors(boundary, horizon):
    a = np.arange(L, boundary - horizon + 1, MEMORY_STRIDE, dtype=np.int64)
    assert len(a) >= TOP_K
    assert int(a.max()) + horizon <= boundary
    return a

def window_multiplicity_weights(boundary, horizon):
    """
    Exact weights induced by flattening and concatenating all valid L-length
    training key windows at stride 8.  This avoids materializing N*L values/channel.
    """
    anchors = valid_memory_anchors(boundary, horizon)
    diff = np.zeros(boundary + 1, np.int32)
    starts = anchors - L
    np.add.at(diff, starts, 1)
    np.add.at(diff, anchors, -1)
    w = np.cumsum(diff[:-1]).astype(np.float32)
    assert w.shape[0] == boundary
    assert w.max() > 0
    return w

def graph_cache_path(dataset, horizon):
    tag = "self1" if GRAPH_INCLUDE_SELF else "self0"
    return GRAPH_ROOT / f"{dataset}_H{horizon}_M{CRAFT_GRAPH_M}_{tag}.npz"

def build_craft_graph(dataset, horizon):
    """
    CRAFT Eq.(1) graph using train-only key windows.
    Returns neighbors[channel] -> M source channels.
    """
    p = graph_cache_path(dataset, horizon)
    if p.exists() and not RECOMPUTE:
        with np.load(p, allow_pickle=False) as zz:
            neigh = np.asarray(zz["neighbors"], np.int64)
            sim = np.asarray(zz["similarity"], np.float32)
        return neigh, sim

    z = DATA[dataset]["z"]
    train_end = DATA[dataset]["train_end"]
    C = DATA[dataset]["channels"]

    w = window_multiplicity_weights(train_end, horizon)
    x = z[:train_end] * np.sqrt(w[:, None], dtype=np.float32)

    # Exact cosine matrix of concatenated historical key windows.
    if torch is not None and torch.cuda.is_available():
        with torch.no_grad():
            xt = torch.from_numpy(x).to("cuda", non_blocking=True)
            gram = xt.T @ xt
            norms = torch.sqrt(torch.clamp(torch.diag(gram), min=1e-12))
            sim_t = gram / (norms[:, None] * norms[None, :] + CRAFT_EPS)
            sim = sim_t.float().cpu().numpy()
            del xt, gram, norms, sim_t
            torch.cuda.empty_cache()
    else:
        gram = x.T @ x
        norms = np.sqrt(np.maximum(np.diag(gram), 1e-12))
        sim = (gram / (norms[:, None] * norms[None, :] + CRAFT_EPS)).astype(np.float32)
        del gram, norms

    sim = np.asarray(sim, np.float32)
    if not GRAPH_INCLUDE_SELF:
        np.fill_diagonal(sim, -np.inf)

    m = min(CRAFT_GRAPH_M, C if GRAPH_INCLUDE_SELF else max(C - 1, 1))
    idx = np.argpartition(-sim, kth=m - 1, axis=1)[:, :m]
    row = np.arange(C)[:, None]
    order = np.argsort(-sim[row, idx], axis=1)
    neighbors = idx[row, order].astype(np.int64)

    np.savez_compressed(
        p,
        neighbors=neighbors,
        similarity=sim,
        graph_m=np.asarray(CRAFT_GRAPH_M, np.int64),
        include_self=np.asarray(int(GRAPH_INCLUDE_SELF), np.int64),
        horizon=np.asarray(horizon, np.int64),
        lookback=np.asarray(L, np.int64),
        stride=np.asarray(MEMORY_STRIDE, np.int64),
    )
    return neighbors, sim

# Quick graph sanity check on ETTh1/H96.
_ng, _gs = build_craft_graph("ETTh1", 96)
print("ETTh1/H96 first graph rows:")
for c in range(min(7, len(_ng))):
    print(c, "->", _ng[c].tolist(), "sim=", [round(float(_gs[c,j]), 4) for j in _ng[c]])


ETTh1/H96 first graph rows:
0 -> [0, 2, 4] sim= [1.0, 0.9836, 0.4032]
1 -> [1, 3, 6] sim= [1.0, 0.9305, 0.6035]
2 -> [2, 0, 4] sim= [1.0, 0.9836, 0.2399]
3 -> [3, 1, 6] sim= [1.0, 0.9305, 0.5232]
4 -> [4, 5, 0] sim= [1.0, 0.4537, 0.4032]
5 -> [5, 4, 6] sim= [1.0, 0.4537, 0.3034]
6 -> [6, 1, 3] sim= [1.0, 0.6035, 0.5232]


In [7]:
# Cell 7 — retrieval evaluation for CRAFT-SC@10 and CRAFT-Graph@10
_SPECTRAL_CACHE = {}

def spectral_memory(dataset, source_ch, horizon, boundary):
    key = (dataset, int(source_ch), int(horizon), int(boundary), CRAFT_F)
    if key in _SPECTRAL_CACHE:
        return _SPECTRAL_CACHE[key]

    z = DATA[dataset]["z"]
    anchors = valid_memory_anchors(boundary, horizon)
    keys = past_windows(z, source_ch, anchors)
    spec = craft_fft_features(keys)

    item = (anchors, spec)
    _SPECTRAL_CACHE[key] = item
    return item

def result_cache_path(dataset, horizon, split, variant):
    tag = "self1" if GRAPH_INCLUDE_SELF else "self0"
    return CACHE_ROOT / f"{dataset}_H{horizon}_{split}_{variant}_F{CRAFT_F}_M{CRAFT_GRAPH_M}_{tag}.npz"

def evaluate_craft_condition(dataset, horizon, split="test", variant="sc", query_chunk=QUERY_CHUNK):
    assert variant in {"sc", "graph"}
    qa, qc = load_manifest(dataset, horizon, split)
    p = result_cache_path(dataset, horizon, split, variant)

    if p.exists() and not RECOMPUTE:
        with np.load(p, allow_pickle=False) as zz:
            ca = np.asarray(zz["anchor"], np.int64)
            cc = np.asarray(zz["channel"], np.int64)
            af10 = np.asarray(zz["analog_mse_top10"], np.float32)
            af1 = np.asarray(zz["analog_mse_top1"], np.float32)
            rf = np.asarray(zz["retrieval_forecast_mse_top10"], np.float32)
            self_frac = np.asarray(zz["top10_same_channel_fraction"], np.float32)
            stride = int(np.asarray(zz["memory_stride"]).item())

        if np.array_equal(ca, qa) and np.array_equal(cc, qc) and stride == MEMORY_STRIDE:
            return {
                "dataset": dataset,
                "horizon": horizon,
                "split": split,
                "variant": variant,
                "pairs": len(qa),
                "analog_future_mse_top10": float(af10.mean()),
                "analog_future_mse_top1": float(af1.mean()),
                "retrieval_forecast_mse_top10": float(rf.mean()),
                "top10_same_channel_fraction": float(self_frac.mean()),
                "seconds": 0.0,
                "cached": True,
            }

    z = DATA[dataset]["z"]
    boundary = DATA[dataset]["train_end"] if split == "val" else DATA[dataset]["val_end"]

    graph = None
    if variant == "graph":
        graph, _ = build_craft_graph(dataset, horizon)

    af10 = np.full(len(qa), np.nan, np.float32)
    af1 = np.full(len(qa), np.nan, np.float32)
    rf = np.full(len(qa), np.nan, np.float32)
    same_frac = np.full(len(qa), np.nan, np.float32)

    t0 = time.perf_counter()

    for qch in np.unique(qc):
        rows = np.flatnonzero(qc == qch)

        if variant == "sc":
            source_channels = np.asarray([qch], np.int64)
        else:
            source_channels = np.asarray(graph[int(qch)], np.int64)

        cand_specs = []
        cand_anchors = []
        cand_channels = []

        for sch in source_channels:
            ma, ms = spectral_memory(dataset, int(sch), horizon, boundary)
            cand_specs.append(ms)
            cand_anchors.append(ma)
            cand_channels.append(np.full(len(ma), int(sch), np.int64))

        kspec = np.concatenate(cand_specs, axis=0)
        kanchor = np.concatenate(cand_anchors, axis=0)
        kchannel = np.concatenate(cand_channels, axis=0)

        assert len(kspec) >= TOP_K
        assert int(kanchor.max()) + horizon <= boundary

        for left in range(0, len(rows), query_chunk):
            rr = rows[left:left + query_chunk]
            qanchor = qa[rr]
            qp = past_windows(z, int(qch), qanchor)
            qspec = craft_fft_features(qp)
            qfuture = future_residuals(z, int(qch), qanchor, horizon)

            sim = craft_similarity(qspec, kspec)

            # Top-10 for the matched AnalogFutureMSE protocol.
            top = np.argpartition(-sim, TOP_K - 1, axis=1)[:, :TOP_K]
            top_scores = np.take_along_axis(sim, top, axis=1)
            order = np.argsort(-top_scores, axis=1)
            top = np.take_along_axis(top, order, axis=1)

            for local_i, global_i in enumerate(rr):
                ids = top[local_i]
                sel_a = kanchor[ids]
                sel_c = kchannel[ids]
                chosen = future_residuals_variable_channel(z, sel_c, sel_a, horizon)
                qf = qfuture[local_i]

                per_candidate = ((chosen - qf[None, :]) ** 2).mean(axis=1)
                af10[global_i] = float(per_candidate.mean())
                af1[global_i] = float(per_candidate[0])

                mean_rel = chosen.mean(axis=0)
                rf[global_i] = float(((mean_rel - qf) ** 2).mean())
                same_frac[global_i] = float(np.mean(sel_c == int(qch)))

        print(
            f"{variant:5s} | {dataset:11s} H={horizon:3d} "
            f"qch={int(qch):3d} pairs={len(rows):4d} "
            f"sources={source_channels.tolist()} candidates={len(kspec):5d}"
        )

    assert np.isfinite(af10).all()
    assert np.isfinite(af1).all()
    assert np.isfinite(rf).all()
    assert np.isfinite(same_frac).all()

    np.savez_compressed(
        p,
        anchor=qa.astype(np.int64),
        channel=qc.astype(np.int16),
        analog_mse_top10=af10,
        analog_mse_top1=af1,
        retrieval_forecast_mse_top10=rf,
        top10_same_channel_fraction=same_frac,
        memory_boundary=np.asarray(boundary, np.int64),
        memory_stride=np.asarray(MEMORY_STRIDE, np.int64),
        top_k=np.asarray(TOP_K, np.int64),
        craft_f=np.asarray(CRAFT_F, np.int64),
        craft_m=np.asarray(CRAFT_GRAPH_M, np.int64),
        graph_include_self=np.asarray(int(GRAPH_INCLUDE_SELF), np.int64),
        variant=np.asarray(variant),
    )

    return {
        "dataset": dataset,
        "horizon": horizon,
        "split": split,
        "variant": variant,
        "pairs": len(qa),
        "analog_future_mse_top10": float(af10.mean()),
        "analog_future_mse_top1": float(af1.mean()),
        "retrieval_forecast_mse_top10": float(rf.mean()),
        "top10_same_channel_fraction": float(same_frac.mean()),
        "seconds": float(time.perf_counter() - t0),
        "cached": False,
    }


In [8]:
# Cell 8 — smoke test before the full 24-condition run
if RUN_SMOKE_TEST:
    print("\n" + "=" * 100)
    print("SMOKE TEST: ETTh1 H=96")
    print("=" * 100)

    smoke_sc = evaluate_craft_condition("ETTh1", 96, "test", "sc")
    print("CRAFT-SC@10   :", smoke_sc)

    if RUN_GRAPH_VARIANT:
        smoke_graph = evaluate_craft_condition("ETTh1", 96, "test", "graph")
        print("CRAFT-Graph@10:", smoke_graph)

    print("[PASS] smoke test completed")



SMOKE TEST: ETTh1 H=96
sc    | ETTh1       H= 96 qch=  0 pairs=1170 sources=[0] candidates= 1417
sc    | ETTh1       H= 96 qch=  1 pairs=1170 sources=[1] candidates= 1417
sc    | ETTh1       H= 96 qch=  2 pairs=1170 sources=[2] candidates= 1417
sc    | ETTh1       H= 96 qch=  3 pairs=1170 sources=[3] candidates= 1417
sc    | ETTh1       H= 96 qch=  4 pairs=1170 sources=[4] candidates= 1417
sc    | ETTh1       H= 96 qch=  5 pairs=1171 sources=[5] candidates= 1417
sc    | ETTh1       H= 96 qch=  6 pairs=1171 sources=[6] candidates= 1417
CRAFT-SC@10   : {'dataset': 'ETTh1', 'horizon': 96, 'split': 'test', 'variant': 'sc', 'pairs': 8192, 'analog_future_mse_top10': 1.765467643737793, 'analog_future_mse_top1': 1.7811033725738525, 'retrieval_forecast_mse_top10': 1.2079603672027588, 'top10_same_channel_fraction': 1.0, 'seconds': 0.7066170452162623, 'cached': False}
graph | ETTh1       H= 96 qch=  0 pairs=1170 sources=[0, 2, 4] candidates= 4251
graph | ETTh1       H= 96 qch=  1 pairs=1170 sour

In [9]:
# Cell 9 — full frozen 24-condition evaluation
rows = []

for d in DATASETS:
    for h in HORIZONS:
        print("\n" + "=" * 100)
        print("CRAFT matched retrieval:", d, "H=", h)
        print("=" * 100)

        r_sc = evaluate_craft_condition(d, h, "test", "sc")
        rows.append(r_sc)
        print(
            f"SC    AFMSE@10={r_sc['analog_future_mse_top10']:.6f} | "
            f"Top1={r_sc['analog_future_mse_top1']:.6f} | "
            f"RFMSE@10={r_sc['retrieval_forecast_mse_top10']:.6f}"
        )

        if RUN_GRAPH_VARIANT:
            r_g = evaluate_craft_condition(d, h, "test", "graph")
            rows.append(r_g)
            print(
                f"Graph AFMSE@10={r_g['analog_future_mse_top10']:.6f} | "
                f"Top1={r_g['analog_future_mse_top1']:.6f} | "
                f"RFMSE@10={r_g['retrieval_forecast_mse_top10']:.6f} | "
                f"same-channel@10={100*r_g['top10_same_channel_fraction']:.1f}%"
            )

        gc.collect()
        if torch is not None and torch.cuda.is_available():
            torch.cuda.empty_cache()

craft_results = (
    pd.DataFrame(rows)
      .sort_values(["variant", "dataset", "horizon"])
      .reset_index(drop=True)
)

expected_n = 24 * (2 if RUN_GRAPH_VARIANT else 1)
assert len(craft_results) == expected_n, (len(craft_results), expected_n)

display(craft_results)
craft_results.to_csv(OUT_ROOT / "craft_frozen24_raw.csv", index=False)



CRAFT matched retrieval: ETTh1 H= 96
SC    AFMSE@10=1.765468 | Top1=1.781103 | RFMSE@10=1.207960
Graph AFMSE@10=1.610380 | Top1=1.556323 | RFMSE@10=1.177803 | same-channel@10=39.2%

CRAFT matched retrieval: ETTh1 H= 192
sc    | ETTh1       H=192 qch=  0 pairs=1170 sources=[0] candidates= 1405
sc    | ETTh1       H=192 qch=  1 pairs=1170 sources=[1] candidates= 1405
sc    | ETTh1       H=192 qch=  2 pairs=1170 sources=[2] candidates= 1405
sc    | ETTh1       H=192 qch=  3 pairs=1170 sources=[3] candidates= 1405
sc    | ETTh1       H=192 qch=  4 pairs=1170 sources=[4] candidates= 1405
sc    | ETTh1       H=192 qch=  5 pairs=1171 sources=[5] candidates= 1405
sc    | ETTh1       H=192 qch=  6 pairs=1171 sources=[6] candidates= 1405
SC    AFMSE@10=1.987178 | Top1=1.942131 | RFMSE@10=1.307169
graph | ETTh1       H=192 qch=  0 pairs=1170 sources=[0, 2, 4] candidates= 4215
graph | ETTh1       H=192 qch=  1 pairs=1170 sources=[1, 3, 6] candidates= 4215
graph | ETTh1       H=192 qch=  2 pairs=1

,dataset,horizon,split,variant,pairs,analog_future_mse_top10,analog_future_mse_top1,retrieval_forecast_mse_top10,top10_same_channel_fraction,seconds,cached
0,ETTh1,96,test,graph,8192,1.610380,1.556323,1.177803,0.392236,0.000000,True
1,ETTh1,192,test,graph,8192,1.774287,1.713030,1.256780,0.386658,1.108911,False
2,ETTh1,336,test,graph,8192,1.901292,1.822481,1.322850,0.385449,1.230292,False
3,ETTh1,720,test,graph,8192,2.174629,2.116523,1.490805,0.390649,1.508352,False
4,Electricity,96,test,graph,8192,1.186141,1.170668,0.951648,0.467053,1.150366,False
5,Electricity,192,test,graph,8192,1.180197,1.158219,0.940815,0.458655,1.215646,False
6,Electricity,336,test,graph,8192,1.229954,1.211617,0.974939,0.466455,1.352831,False
7,Electricity,720,test,graph,8192,1.270072,1.256732,0.994101,0.460474,1.624419,False
8,Exchange,96,test,graph,8192,0.228074,0.222646,0.104542,0.441797,0.729507,False
9,Exchange,192,test,graph,8192,0.501801,0.499643,0.222429,0.450122,0.819508,False


In [10]:
# Cell 10 — exact frozen Pearson / PRR / Anchored-L2 references
# Prefer the user's already generated local result CSVs; otherwise use the paper-aligned values.

l2_csv = PROJECT_ROOT / "results" / "anchored_l2_long_horizon_frozen24" / "anchored_l2_test_24conditions.csv"
precise_csv = PROJECT_ROOT / "results" / "stride8_revalidation" / "table1_stride8_revalidated.csv"

if l2_csv.exists():
    l2 = (
        pd.read_csv(l2_csv)[["dataset", "horizon", "anchored_l2_analog_future_mse"]]
          .rename(columns={"anchored_l2_analog_future_mse": "anchored_l2"})
    )
else:
    l2 = pd.DataFrame([
        ("ETTh1",96,1.100985),("ETTh1",192,1.277228),("ETTh1",336,1.446432),("ETTh1",720,1.714012),
        ("Electricity",96,.703011),("Electricity",192,.728439),("Electricity",336,.768702),("Electricity",720,.815970),
        ("Exchange",96,.186254),("Exchange",192,.379211),("Exchange",336,.684537),("Exchange",720,1.865029),
        ("Solar",96,.488054),("Solar",192,.518212),("Solar",336,.533977),("Solar",720,.527471),
        ("Traffic",96,1.420632),("Traffic",192,1.355879),("Traffic",336,1.424405),("Traffic",720,1.595257),
        ("Weather",96,.839593),("Weather",192,.785080),("Weather",336,.594079),("Weather",720,.769931),
    ], columns=["dataset", "horizon", "anchored_l2"])

if precise_csv.exists():
    p = pd.read_csv(precise_csv)
    need = {"dataset", "horizon", "pearson_analog_future_mse", "prr_analog_future_mse"}
    assert need.issubset(p.columns), p.columns
    ref = (
        p[["dataset", "horizon", "pearson_analog_future_mse", "prr_analog_future_mse"]]
        .rename(columns={
            "pearson_analog_future_mse": "pearson",
            "prr_analog_future_mse": "prr",
        })
    )
else:
    ref = pd.DataFrame([
        ("ETTh1",96,1.612267,1.1643),("ETTh1",192,1.817531,1.2229),("ETTh1",336,1.992872,1.4816),("ETTh1",720,2.254092,1.4618),
        ("Weather",96,1.161329,.4316),("Weather",192,1.062809,.5147),("Weather",336,1.290959,.6748),("Weather",720,1.295817,.8130),
        ("Electricity",96,1.179589,.7191),("Electricity",192,1.195698,.7345),("Electricity",336,1.241844,.7648),("Electricity",720,1.287729,.8602),
        ("Traffic",96,1.973787,1.4696),("Traffic",192,2.004524,1.4508),("Traffic",336,2.063635,1.5147),("Traffic",720,2.166454,1.7351),
        ("Exchange",96,.213266,.1652),("Exchange",192,.423408,.4116),("Exchange",336,.764282,.8094),("Exchange",720,1.886809,1.7770),
        ("Solar",96,.588674,.5087),("Solar",192,.597890,.5406),("Solar",336,.623836,.5701),("Solar",720,.638530,.5507),
    ], columns=["dataset", "horizon", "pearson", "prr"])

baseline = ref.merge(l2, on=["dataset", "horizon"], how="inner")
baseline["horizon"] = baseline["horizon"].astype(int)
assert len(baseline) == 24
display(baseline.sort_values(["dataset", "horizon"]))


,dataset,horizon,pearson,prr,anchored_l2
0,ETTh1,96,1.612267,1.1643,1.100985
1,ETTh1,192,1.817531,1.2229,1.277228
2,ETTh1,336,1.992872,1.4816,1.446432
3,ETTh1,720,2.254092,1.4618,1.714012
4,Electricity,96,1.179589,0.7191,0.703011
5,Electricity,192,1.195698,0.7345,0.728439
6,Electricity,336,1.241844,0.7648,0.768702
7,Electricity,720,1.287729,0.8602,0.815970
8,Exchange,96,0.213266,0.1652,0.186254
9,Exchange,192,0.423408,0.4116,0.379211


In [11]:
# Cell 11 — matched comparisons, including CRAFT-paper-overlap subset
wide = (
    craft_results
      .pivot(index=["dataset", "horizon"], columns="variant", values="analog_future_mse_top10")
      .reset_index()
      .rename(columns={"sc": "craft_sc", "graph": "craft_graph"})
)

comp = (
    baseline.merge(wide, on=["dataset", "horizon"], how="inner")
            .sort_values(["dataset", "horizon"])
            .reset_index(drop=True)
)

assert len(comp) == 24

def add_gain(df, method, baseline_name):
    df[f"{method}_gain_vs_{baseline_name}_pct"] = (
        100.0 * (df[baseline_name] - df[method]) / df[baseline_name]
    )

for method in ["craft_sc"] + (["craft_graph"] if "craft_graph" in comp.columns else []):
    for b in ["pearson", "anchored_l2", "prr"]:
        add_gain(comp, method, b)

def wtl(x, tol=1e-12):
    x = np.asarray(x, float)
    return {
        "W": int((x > tol).sum()),
        "T": int((np.abs(x) <= tol).sum()),
        "L": int((x < -tol).sum()),
        "mean_gain_pct": float(x.mean()),
        "median_gain_pct": float(np.median(x)),
    }

def summarize_scope(df, scope_name):
    rows = []
    for method, label in [
        ("craft_sc", "CRAFT-SC@10"),
        ("craft_graph", "CRAFT-Graph@10"),
    ]:
        if method not in df.columns:
            continue
        for b, blabel in [
            ("pearson", "Pearson"),
            ("anchored_l2", "Anchored L2"),
            ("prr", "PRR"),
        ]:
            g = 100.0 * (df[b] - df[method]) / df[b]
            rows.append({
                "scope": scope_name,
                "comparison": f"{label} vs {blabel}",
                **wtl(g),
            })
    return pd.DataFrame(rows)

summary_all = summarize_scope(comp, "All six datasets / 24")
summary_overlap = summarize_scope(
    comp[comp["dataset"].isin(CRAFT_PAPER_OVERLAP)].copy(),
    "CRAFT-paper overlap / 16",
)
summary = pd.concat([summary_overlap, summary_all], ignore_index=True)

display(comp)
display(summary)

comp.to_csv(OUT_ROOT / "craft_matched_exact_24.csv", index=False)
summary.to_csv(OUT_ROOT / "craft_matched_summary.csv", index=False)


,dataset,horizon,pearson,prr,anchored_l2,craft_graph,craft_sc,craft_sc_gain_vs_pearson_pct,craft_sc_gain_vs_anchored_l2_pct,craft_sc_gain_vs_prr_pct,craft_graph_gain_vs_pearson_pct,craft_graph_gain_vs_anchored_l2_pct,craft_graph_gain_vs_prr_pct
0,ETTh1,96,1.612267,1.1643,1.100985,1.610380,1.765468,-9.502187,-60.353427,-51.633397,0.117053,-46.267151,-38.313134
1,ETTh1,192,1.817531,1.2229,1.277228,1.774287,1.987178,-9.333947,-55.585165,-62.497166,2.379259,-38.916956,-45.088458
2,ETTh1,336,1.992872,1.4816,1.446432,1.901292,2.127021,-6.731457,-47.052949,-43.562438,4.595374,-31.447016,-28.326934
3,ETTh1,720,2.254092,1.4618,1.714012,2.174629,2.376473,-5.429290,-38.649762,-62.571722,3.525288,-26.873621,-48.763783
4,Electricity,96,1.179589,0.7191,0.703011,1.186141,1.242946,-5.371159,-76.803374,-72.847502,-0.555466,-68.723072,-64.947993
5,Electricity,192,1.195698,0.7345,0.728439,1.180197,1.249896,-4.532691,-71.585407,-70.169597,1.296409,-62.017218,-60.680359
6,Electricity,336,1.241844,0.7648,0.768702,1.229954,1.298534,-4.564975,-68.925491,-69.787353,0.957404,-60.004048,-60.820393
7,Electricity,720,1.287729,0.8602,0.815970,1.270072,1.328092,-3.134402,-62.762354,-54.393339,1.371195,-55.651811,-47.648410
8,Exchange,96,0.213266,0.1652,0.186254,0.228074,0.218456,-2.433655,-17.289314,-32.237541,-6.943503,-22.453213,-38.059565
9,Exchange,192,0.423408,0.4116,0.379211,0.501801,0.483368,-14.161223,-27.466697,-17.436377,-18.514593,-32.327451,-21.914640


,scope,comparison,W,T,L,mean_gain_pct,median_gain_pct
0,CRAFT-paper overlap / 16,CRAFT-SC@10 vs Pearson,4,0,12,-4.267919,-5.400225
1,CRAFT-paper overlap / 16,CRAFT-SC@10 vs Anchored L2,1,0,15,-57.941611,-65.843923
2,CRAFT-paper overlap / 16,CRAFT-SC@10 vs PRR,0,0,16,-62.230699,-66.179537
3,CRAFT-paper overlap / 16,CRAFT-Graph@10 vs Pearson,11,0,5,1.837460,1.333802
4,CRAFT-paper overlap / 16,CRAFT-Graph@10 vs Anchored L2,1,0,15,-48.072852,-56.421955
5,CRAFT-paper overlap / 16,CRAFT-Graph@10 vs PRR,0,0,16,-52.592685,-57.169608
6,All six datasets / 24,CRAFT-SC@10 vs Pearson,8,0,16,-2.459367,-4.548833
7,All six datasets / 24,CRAFT-SC@10 vs Anchored L2,1,0,23,-42.788160,-42.851355
8,All six datasets / 24,CRAFT-SC@10 vs PRR,0,0,24,-44.456330,-54.930842
9,All six datasets / 24,CRAFT-Graph@10 vs Pearson,16,0,8,2.510636,1.875227


In [12]:
# Cell 12 — dataset-level diagnostic and graph-support statistics
dataset_rows = []

graph_support = (
    craft_results[craft_results["variant"] == "graph"]
    [["dataset", "horizon", "top10_same_channel_fraction"]]
    if RUN_GRAPH_VARIANT else pd.DataFrame()
)

for d, g in comp.groupby("dataset", sort=False):
    row = {"dataset": d, "conditions": len(g)}

    for method, label in [("craft_sc", "SC"), ("craft_graph", "Graph")]:
        if method not in g.columns:
            continue

        vs_prr = 100.0 * (g["prr"] - g[method]) / g["prr"]
        vs_l2 = 100.0 * (g["anchored_l2"] - g[method]) / g["anchored_l2"]

        a = wtl(vs_prr)
        b = wtl(vs_l2)

        row[f"{label}_vs_PRR_W"] = a["W"]
        row[f"{label}_vs_PRR_T"] = a["T"]
        row[f"{label}_vs_PRR_L"] = a["L"]
        row[f"{label}_vs_PRR_mean_pct"] = a["mean_gain_pct"]

        row[f"{label}_vs_L2_W"] = b["W"]
        row[f"{label}_vs_L2_T"] = b["T"]
        row[f"{label}_vs_L2_L"] = b["L"]
        row[f"{label}_vs_L2_mean_pct"] = b["mean_gain_pct"]

    if RUN_GRAPH_VARIANT:
        sg = graph_support[graph_support["dataset"] == d]
        row["Graph_top10_same_channel_pct"] = (
            100.0 * float(sg["top10_same_channel_fraction"].mean())
        )

    dataset_rows.append(row)

dataset_summary = pd.DataFrame(dataset_rows)
display(dataset_summary)
dataset_summary.to_csv(OUT_ROOT / "craft_matched_dataset_summary.csv", index=False)


,dataset,conditions,SC_vs_PRR_W,SC_vs_PRR_T,SC_vs_PRR_L,SC_vs_PRR_mean_pct,SC_vs_L2_W,SC_vs_L2_T,SC_vs_L2_L,SC_vs_L2_mean_pct,Graph_vs_PRR_W,Graph_vs_PRR_T,Graph_vs_PRR_L,Graph_vs_PRR_mean_pct,Graph_vs_L2_W,Graph_vs_L2_T,Graph_vs_L2_L,Graph_vs_L2_mean_pct,Graph_top10_same_channel_pct
0,ETTh1,4,0,0,4,-55.066181,0,0,4,-50.410326,0,0,4,-40.123077,0,0,4,-35.876186,38.874818
1,Electricity,4,0,0,4,-66.799448,0,0,4,-70.019156,0,0,4,-58.524289,0,0,4,-61.599037,46.315920
2,Exchange,4,0,0,4,-14.682681,0,0,4,-16.757540,2,0,2,-14.116095,1,0,3,-16.220974,45.885850
3,Solar,4,0,0,4,-3.132499,0,0,4,-8.204980,4,0,0,2.094733,0,0,4,-2.719549,33.298340
4,Traffic,4,0,0,4,-71.011087,0,0,4,-81.854816,0,0,4,-66.667840,0,0,4,-77.243257,42.799378
5,Weather,4,0,0,4,-56.046082,1,0,3,-29.482145,0,0,4,-45.055534,1,0,3,-17.572929,45.182802


In [13]:
# Cell 13 — paper-facing result block
print("\n" + "=" * 108)
print("PAPER-FACING CRAFT MATCHED RETRIEVAL RESULT")
print("=" * 108)
print(
    f"Protocol: L={L}, stride={MEMORY_STRIDE}, TopK={TOP_K}, "
    f"CRAFT F={CRAFT_F} ({100*CRAFT_FREQ_RATIO:.1f}% of L), "
    f"graph M={CRAFT_GRAPH_M}, self={'included' if GRAPH_INCLUDE_SELF else 'excluded'}"
)
print("IMPORTANT: matched retrieval-rule diagnostic, not end-to-end CRAFT forecasting reproduction.")

for scope in ["CRAFT-paper overlap / 16", "All six datasets / 24"]:
    print("\n" + scope)
    print("-" * len(scope))
    ss = summary[summary["scope"] == scope]
    for r in ss.itertuples(index=False):
        print(
            f"{r.comparison:32s}: "
            f"{r.W}W/{r.T}T/{r.L}L | "
            f"mean={r.mean_gain_pct:+.3f}% | median={r.median_gain_pct:+.3f}%"
        )

if RUN_GRAPH_VARIANT:
    mean_self = 100.0 * float(
        craft_results.loc[
            craft_results["variant"] == "graph",
            "top10_same_channel_fraction",
        ].mean()
    )
    print(f"\nCRAFT-Graph mean same-channel fraction in final Top-10: {mean_self:.1f}%")

print("\nSaved under:", OUT_ROOT)
print("\nFiles:")
for p in sorted(OUT_ROOT.glob("*.csv")):
    print(" -", p.name)



PAPER-FACING CRAFT MATCHED RETRIEVAL RESULT
Protocol: L=96, stride=8, TopK=10, CRAFT F=5 (5.0% of L), graph M=3, self=included
IMPORTANT: matched retrieval-rule diagnostic, not end-to-end CRAFT forecasting reproduction.

CRAFT-paper overlap / 16
------------------------
CRAFT-SC@10 vs Pearson          : 4W/0T/12L | mean=-4.268% | median=-5.400%
CRAFT-SC@10 vs Anchored L2      : 1W/0T/15L | mean=-57.942% | median=-65.844%
CRAFT-SC@10 vs PRR              : 0W/0T/16L | mean=-62.231% | median=-66.180%
CRAFT-Graph@10 vs Pearson       : 11W/0T/5L | mean=+1.837% | median=+1.334%
CRAFT-Graph@10 vs Anchored L2   : 1W/0T/15L | mean=-48.073% | median=-56.422%
CRAFT-Graph@10 vs PRR           : 0W/0T/16L | mean=-52.593% | median=-57.170%

All six datasets / 24
---------------------
CRAFT-SC@10 vs Pearson          : 8W/0T/16L | mean=-2.459% | median=-4.549%
CRAFT-SC@10 vs Anchored L2      : 1W/0T/23L | mean=-42.788% | median=-42.851%
CRAFT-SC@10 vs PRR              : 0W/0T/24L | mean=-44.456% | med

## Interpretation guide for the manuscript

Use the result only after the full run is complete.

**If PRR beats both CRAFT-SC@10 and CRAFT-Graph@10 on most overlap conditions:**  
this directly answers the reviewer concern that the Pearson comparison is too weak. Add a compact matched-retrieval table to Appendix G and one sentence in Sec. 5.2.

**If CRAFT-SC is competitive but CRAFT-Graph is weaker/stronger:**  
report both. They answer different questions: CRAFT-SC is the apples-to-apples scoring-rule comparison; CRAFT-Graph tests the recent method's cross-channel support mechanism.

**If CRAFT beats PRR substantially:**  
do not hide it. Keep Pearson as the controlled main baseline, report CRAFT transparently in Appendix G, and frame PRR as a future-supervised alternative whose relative advantage is domain-dependent.

Do **not** call this an exact reproduction of CRAFT's forecasting benchmark. The original paper uses \(L=720\), \(k=1\), a direct forecaster, retrieved-reference projection, and an additive coefficient \(\alpha\). This notebook intentionally holds the PRR retrieval benchmark fixed to isolate which histories are selected.
